In [1]:
def read_conllu(file_path):
    sentences = []
    labels = []

    with open(file_path, "r", encoding="utf-8") as f:
        words, tags = [], []

        for line in f:
            if line.strip() == "":
                if words:
                    sentences.append(words)
                    labels.append(tags)
                    words, tags = [], []
                continue

            if line.startswith("#"):
                continue

            parts = line.strip().split("\t")

            if "-" in parts[0] or "." in parts[0]:
                continue  # skip special tokens

            words.append(parts[1])
            tags.append(parts[3])  # UPOS

    return sentences, labels


train_sentences, train_labels = read_conllu("mr_ufal-ud-train.conllu")
val_sentences, val_labels = read_conllu("mr_ufal-ud-dev.conllu")
test_sentences, test_labels = read_conllu("mr_ufal-ud-test.conllu")

print(train_sentences[0])
print(train_labels[0])

['एक', 'होता', 'राजा', '.']
['DET', 'AUX', 'NOUN', 'PUNCT']


In [2]:
label_list = sorted(list(set(tag for sent in train_labels for tag in sent)))

label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

print(label_list)

['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'VERB', 'X']


In [3]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "tokens": train_sentences,
    "labels": train_labels
})

val_dataset = Dataset.from_dict({
    "tokens": val_sentences,
    "labels": val_labels
})

test_dataset = Dataset.from_dict({
    "tokens": test_sentences,
    "labels": test_labels
})

/Users/tejasmankeshwar/Developer/College/NLP-MiniProject/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True
    )

    word_ids = tokenized_inputs.word_ids()
    label_ids = []
    previous_word_idx = None

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[example["labels"][word_idx]])
        else:
            label_ids.append(-100)

        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs


train_dataset = train_dataset.map(tokenize_and_align_labels)
val_dataset = val_dataset.map(tokenize_and_align_labels)
test_dataset = test_dataset.map(tokenize_and_align_labels)

Map: 100%|██████████| 47/47 [00:00<00:00, 8403.27 examples/s]


In [7]:
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    save_strategy="epoch",
    logging_dir="./logs"
)

# 🔥 IMPORTANT FIX: Data collator (handles padding properly)
data_collator = DataCollatorForTokenClassification(tokenizer)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator   # ✅ FIX ADDED
)

# Train
trainer.train()

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 7205.58it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead

Step,Training Loss


RuntimeError: MPS backend out of memory (MPS allocated: 5.96 GiB, other allocations: 3.59 GiB, max allowed: 9.07 GiB). Tried to allocate 256 bytes on shared pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
import transformers
print(transformers.__version__)

5.6.2
